#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when, count, to_date, length, mean, right
from pyspark.sql import Window

In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos

def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()
    
    return

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.cust_az12")

In [0]:
info(df) 

#Transformations

In [0]:
df.show()

## Rename columns' names

In [0]:
RENAME_MAP = {
    "CID": "cliente_number",
    "BDATE": "birth_date",
    "GEN": "gender"
   
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimm

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Unique Values

In [0]:
info(df)
print("", "Não há duplicações")


## Missings Values

### Gender

In [0]:
#Identificando as colunas com Vazios
info(df)



In [0]:
#Entendendo os Casos
df.select("Gender").distinct().show()



In [0]:
#Tratando os Casos
df_alt = df.withColumn(
    "gender",
    when(F.upper(F.col("gender")) == "M","Male")
    .when(F.upper(F.col("gender")) == "F","Female")
    .when(col("gender").isNull(),"Unknown")
    .when(col("gender") == "","Unknown")
    .otherwise(col("gender"))
    )
#


#Validando o tratamento
info(df_alt)
df_alt.select("Gender").distinct().show()


## New Columns

In [0]:
%skip
#Analisar a coluna cliente_number

cnb = df_alt.withColumn("Len_cnb", length(col("cliente_number")))

cnb.groupBy("Len_cnb").count().show()

print("","Analisando os casos com 10")

cnb.filter(col("Len_cnb") == 10).show(n=1)
cnb.filter(col("Len_cnb") == 13).show(n=1)

cnb1 = (cnb.withColumn("cliente_number", F.substring(col("cliente_number"), -10,10))
        .withColumn("Len_cnb", length(col("cliente_number")))
)
cnb1.groupBy("Len_cnb").count().show()



In [0]:
#Criar colunas customer_id e customer_number
df_alt = (
    df_alt.withColumn("customer_id",F.substring(col("cliente_number"),-10,10)).
withColumn("customer_number", F.substring(col("cliente_number"),-5,5))
)

#Validando novas Colunas

df_alt.show(n=5)

info(df_alt)

#Write into Silver Layer

In [0]:
df_alt.write.mode("overwrite").saveAsTable("workspace.silver.erp_customers")
